In [1]:
%load_ext autoreload
%autoreload 2

# Imports

In [37]:
import numpy as np
import copy
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler, WeightedRandomSampler


In [51]:
from datasets import *
from tokenizer import LogTokenizer
from utils import get_padded_data
from model.loss_function import SimpleLossCompute
from model.model import LogsyModel
from model.trainer import run_train, run_test

In [39]:
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve

### BGL Dataset

In [4]:
log_format = '<t> <Timestamp> <Date> <Node> <Time> <NodeRepeat> <Type> <Component> <Level> <Content>'  #BGL
every_n = 1
aux = 300000

bgl_dataset = BGLDataset(log_format=log_format, every_n=every_n, aux_size=aux)

bgl_aux_normal, bgl_aux_anomalies = bgl_dataset.get_data()

(4713493, 11)


### Spirit2 Dataset

In [5]:
log_format = '<t> <Timestamp> <Date> <User> <Month> <Day> <Time> <Location> <Content>'  #spirit2
every_n = 100
aux_size = 100000

spirit_dataset = SpiritDataset(log_format=log_format, every_n=every_n, aux_size=aux)

spirit_aux_normal, spirit_aux_anomalies = spirit_dataset.get_data()

(2722990, 10)


### Intrepid_RAS_0901_0908_scrubbed Dataset

In [6]:
log_format = '<f> <a>          <c>       <d>                  <e>    <t> <Content>'  #BGL
every_n = 1

intrepid_dataset = InterpidScrubbedDataset(log_format=log_format, every_n=every_n)

aux_anomalies_t, tl = intrepid_dataset.get_data_a()

# the labels are inverted in this dataset
inter_aux_anomalies = aux_anomalies_t[tl==0]
inter_aux_normal = aux_anomalies_t[tl==1]

### Thunderbird Dataset

In [7]:
log_format = '<t> <Timestamp> <Date> <User> <Month> <Day> <Time> <Location> <Component>(\[<PID>\])?: <Content>'  #thunderbird
every_n = 1
max_lines = 5000000

tbird_dataset = ThunderbirdDataset(log_format=log_format, every_n=every_n,max_lines=max_lines)

log_payload, true_labels = tbird_dataset.get_data()
true_labels = true_labels.reshape(-1,1)

log_payload_original = copy.deepcopy(log_payload)
true_labels_original = copy.deepcopy(true_labels)

(4992129, 12)


In [24]:
df_size = len(log_payload_original)
df_size

4992129

### Concatenate auxilary datasets

In [10]:
aux_anomalies = np.append(np.append(bgl_aux_anomalies,spirit_aux_anomalies),inter_aux_anomalies)
print(aux_anomalies.shape)

(633370,)


In [11]:
log_payload = np.append(log_payload.values.reshape(-1,1), aux_anomalies.reshape(-1,1), axis=0)
true_labels = np.append(true_labels,  np.ones(len(aux_anomalies)).reshape(-1,1), axis=0).flatten()

print(log_payload.shape, true_labels.shape)

(5625499, 1) (5625499,)


## Tokenize the corpus

In [25]:
tokenizer = LogTokenizer()
data_tokenized = []
for i in range(0, len(log_payload)):
    tokenized = tokenizer.tokenize(log_payload[i][0])
    data_tokenized.append(tokenized)

### Index the data

In [29]:
data_token_indexed = np.asanyarray(data_tokenized)

## Train-Test Split

In [30]:
ratio = 0.8
train_size = round(df_size * ratio)
print(df_size, train_size)

4992129 3993703


In [31]:
data_token_indexed_test = data_token_indexed[train_size:df_size]
data_token_indexed_train = np.append(data_token_indexed[:train_size][true_labels[:train_size]==0], 
                                     data_token_indexed[df_size:],axis=0)

test_ground_labels = true_labels[train_size:df_size]
train_ground_labels = np.append(true_labels[:train_size][true_labels[:train_size]==0].flatten(), 
                                true_labels[df_size:].flatten(),axis=0)

In [34]:
batch_size = 2048
transform_to_tensor = transforms.Lambda(lambda lst: torch.tensor(lst))
train_data = TensorDataset(torch.tensor(get_padded_data(data_token_indexed_train, pad_len=50), dtype=torch.long), torch.tensor(train_ground_labels.astype(np.int32), dtype=torch.long))
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

test_data = TensorDataset(torch.tensor(get_padded_data(data_token_indexed_test, pad_len=50), dtype=torch.long), torch.tensor(test_ground_labels.astype(np.int32).flatten(), dtype=torch.long))
test_sampler = SequentialSampler(test_data)
test_dataloader = DataLoader(test_data, sampler=test_sampler, batch_size=batch_size)


# Create model

In [82]:
src_vocab=tokenizer.n_words
tgt_vocab=2
n_layers=2
in_features=16
out_features=16
h=2
dropout=0.05
max_len=50

model = LogsyModel(src_vocab, tgt_vocab, n_layers, in_features, out_features, h, dropout, max_len).get_model()

EncoderDecoder(
  (encoder): Encoder(
    (layers): ModuleList(
      (0): EncoderLayer(
        (self_attn): MultiHeadedAttention(
          (linears): ModuleList(
            (0): Linear(in_features=16, out_features=16, bias=True)
            (1): Linear(in_features=16, out_features=16, bias=True)
            (2): Linear(in_features=16, out_features=16, bias=True)
            (3): Linear(in_features=16, out_features=16, bias=True)
          )
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (feed_forward): PositionwiseFeedForward(
          (w_1): Linear(in_features=16, out_features=16, bias=True)
          (w_2): Linear(in_features=16, out_features=16, bias=True)
          (dropout): Dropout(p=0.05, inplace=False)
        )
        (sublayer): ModuleList(
          (0): SublayerConnection(
            (norm): LayerNorm()
            (dropout): Dropout(p=0.05, inplace=False)
          )
          (1): SublayerConnection(
            (norm): LayerNorm()
           

### Initialize loss and optimizer

In [83]:
#loss
criterion = nn.CrossEntropyLoss(weight=torch.tensor([0.3,1.0]).cuda())#
model_opt = torch.optim.Adam(model.parameters(), lr=0.0001, betas=(0.9, 0.999), weight_decay=0.001)

In [ ]:
max_auc = 0.0
max_distances = 0
for epoch in range(30):
    model.train()
    print("Epoch",epoch)
    run_train(train_dataloader, model, 
             SimpleLossCompute(model, criterion, model_opt), step_size=100)
    torch.save(model.state_dict(), '../output/models/model_'+str(epoch)+'.pt')
    
    
    #test
    model.eval()
    preds, distances = run_test(test_dataloader, model, 
                        SimpleLossCompute(model, criterion, None, is_test=True), step_size=100)

    preds = np.array(preds)
    auc = roc_auc_score(test_ground_labels.astype(np.int32), distances)
    print("AUC:", auc)
    if auc > max_auc:
        max_auc = auc
        fpr, tpr, thresholds = roc_curve(test_ground_labels.astype(np.int32), distances, pos_label=1)
        np.save(str(aux_size)+'_without8020.npy',[fpr, tpr, thresholds])
        print(roc_auc_score(test_ground_labels.astype(np.int32), distances))
        max_distances = distances
    

Epoch 0
Epoch Step: 1 / 2180 Loss: 3.268634
Epoch Step: 101 / 2180 Loss: 3.249446
Epoch Step: 201 / 2180 Loss: 3.177441
Epoch Step: 301 / 2180 Loss: 3.124229
Epoch Step: 401 / 2180 Loss: 3.000609
Epoch Step: 501 / 2180 Loss: 2.933898
Epoch Step: 601 / 2180 Loss: 2.884660
Epoch Step: 701 / 2180 Loss: 2.802370
Epoch Step: 801 / 2180 Loss: 2.680750
Epoch Step: 901 / 2180 Loss: 2.595043
Epoch Step: 1001 / 2180 Loss: 2.558646
Epoch Step: 1101 / 2180 Loss: 2.476412
Epoch Step: 1201 / 2180 Loss: 2.447673
Epoch Step: 1301 / 2180 Loss: 2.271137
Epoch Step: 1401 / 2180 Loss: 2.286556
Epoch Step: 1501 / 2180 Loss: 2.217210
Epoch Step: 1601 / 2180 Loss: 2.181738
Epoch Step: 1701 / 2180 Loss: 2.087662
Epoch Step: 1801 / 2180 Loss: 1.999464
Epoch Step: 1901 / 2180 Loss: 1.954398
Epoch Step: 2001 / 2180 Loss: 1.929347
Epoch Step: 2101 / 2180 Loss: 1.886674
Epoch Step: 1 / 488 Loss: 1.979331
Epoch Step: 101 / 488 Loss: 2.060996
Epoch Step: 201 / 488 Loss: 1.992939
Epoch Step: 301 / 488 Loss: 2.008496


Epoch Step: 1901 / 2180 Loss: 0.007730
Epoch Step: 2001 / 2180 Loss: 0.008701
Epoch Step: 2101 / 2180 Loss: 0.007131
Epoch Step: 1 / 488 Loss: 0.283096
Epoch Step: 101 / 488 Loss: 0.118327
Epoch Step: 201 / 488 Loss: 0.255676
Epoch Step: 301 / 488 Loss: 0.224315
Epoch Step: 401 / 488 Loss: 0.330150
AUC: 0.9997113872090431
Epoch 8
Epoch Step: 1 / 2180 Loss: 0.007655
Epoch Step: 101 / 2180 Loss: 0.010531
Epoch Step: 201 / 2180 Loss: 0.009500
Epoch Step: 301 / 2180 Loss: 0.007715
Epoch Step: 401 / 2180 Loss: 0.007679
Epoch Step: 501 / 2180 Loss: 0.008610
Epoch Step: 601 / 2180 Loss: 0.011940
Epoch Step: 701 / 2180 Loss: 0.009887
Epoch Step: 801 / 2180 Loss: 0.008559
Epoch Step: 901 / 2180 Loss: 0.006972
Epoch Step: 1001 / 2180 Loss: 0.008118
Epoch Step: 1101 / 2180 Loss: 0.006890
Epoch Step: 1201 / 2180 Loss: 0.006946
Epoch Step: 1301 / 2180 Loss: 0.009574
Epoch Step: 1401 / 2180 Loss: 0.007019
Epoch Step: 1501 / 2180 Loss: 0.008086
Epoch Step: 1601 / 2180 Loss: 0.007504
Epoch Step: 1701 

Epoch Step: 1201 / 2180 Loss: 0.005984
Epoch Step: 1301 / 2180 Loss: 0.005948
Epoch Step: 1401 / 2180 Loss: 0.007508
Epoch Step: 1501 / 2180 Loss: 0.006114
Epoch Step: 1601 / 2180 Loss: 0.009528
Epoch Step: 1701 / 2180 Loss: 0.009920
Epoch Step: 1801 / 2180 Loss: 0.008052
Epoch Step: 1901 / 2180 Loss: 0.006103
Epoch Step: 2001 / 2180 Loss: 0.007738
Epoch Step: 2101 / 2180 Loss: 0.007174
Epoch Step: 1 / 488 Loss: 0.395047
Epoch Step: 101 / 488 Loss: 0.165051
Epoch Step: 201 / 488 Loss: 0.356741
Epoch Step: 301 / 488 Loss: 0.312957
Epoch Step: 401 / 488 Loss: 0.460772
AUC: 0.99904010436907
Epoch 16
Epoch Step: 1 / 2180 Loss: 0.007814
Epoch Step: 101 / 2180 Loss: 0.007309
Epoch Step: 201 / 2180 Loss: 0.007339
Epoch Step: 301 / 2180 Loss: 0.006150
Epoch Step: 401 / 2180 Loss: 0.007321
Epoch Step: 501 / 2180 Loss: 0.006404
Epoch Step: 601 / 2180 Loss: 0.006799
Epoch Step: 701 / 2180 Loss: 0.007385
Epoch Step: 801 / 2180 Loss: 0.007308
Epoch Step: 901 / 2180 Loss: 0.005521
Epoch Step: 1001 /

In [65]:
model.model.generator

Generator(
  (proj): Linear(in_features=16, out_features=2, bias=True)
)